# SmolVLM2 MELD Benchmark

Kaggle pipeline for extracting frozen SmolVLM2 shared video+utterance embeddings, training the same MLP probe used by the other model pipelines, and inspecting labeled metrics. The benchmark defaults to 6 FPS; change only `FPS` in the configuration cell to run the official 1 FPS temporal setting.

In [ ]:
# Install the repository requirements. Restart the session only if Kaggle asks.
CODE = "/kaggle/input/datasets/pushkarsingh2005/benchmark-code"
!pip install -q -r {CODE}/requirements.txt

In [ ]:
# Experiment configuration: change FPS to 1.0 for SmolVLM2's official temporal default.
from pathlib import Path

MODEL_ID = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
FPS = 6.0
MAX_FRAMES = 64
CHUNK_SIZE = 500
POOLING = "last"
PROMPT_STYLE = "emotion_task"
MODALITY_MODE = "video_text"
SAVE_DTYPE = "float32"

INDEX_DIR = Path(CODE) / "outputs/indexes"
FPS_TAG = f"fps{FPS:g}".replace(".", "p")
RUN_NAME = f"smolvlm2_2_2b_shared_{FPS_TAG}"
EMB_ROOT = Path("/kaggle/working/embeddings") / RUN_NAME
CHUNK_ROOT = Path("/kaggle/working/embeddings") / f"{RUN_NAME}_chunks"
OUT_DIR = Path("/kaggle/working/outputs") / RUN_NAME

print("Run:", RUN_NAME)
print("Sampling:", FPS, "FPS, at most", MAX_FRAMES, "frames")
print("Chunks:", CHUNK_ROOT)
print("Merged embeddings:", EMB_ROOT)
print("MLP results:", OUT_DIR)

In [ ]:
# Runtime check
import torch
import transformers

print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a GPU accelerator in Kaggle before extraction."
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

In [ ]:
# Check that the prepared MELD indexes are available and point to videos.
import pandas as pd

for split in ["train", "dev", "test"]:
    path = INDEX_DIR / f"meld_{split}_index.csv"
    df = pd.read_csv(path)
    existing = int(df["video_exists"].astype(bool).sum()) if "video_exists" in df else "not recorded"
    print(f"{split}: rows={len(df)}, videos marked present={existing}")
    display(df[["sample_id", "video_path", "utterance", "emotion"]].head(2))

In [ ]:
# Extract resumable chunks. Each chunk runs in a fresh process to release CPU/GPU memory.
import subprocess
import sys

for split in ["train", "dev", "test"]:
    cmd = [
        sys.executable, str(Path(CODE) / "scripts/run_smolvlm2_chunks.py"),
        "--index-csv", str(INDEX_DIR / f"meld_{split}_index.csv"),
        "--chunks-dir", str(CHUNK_ROOT / split),
        "--chunk-prefix", split,
        "--chunk-size", str(CHUNK_SIZE),
        "--model-id", MODEL_ID,
        "--fps", str(FPS),
        "--max-frames", str(MAX_FRAMES),
        "--pooling", POOLING,
        "--prompt-style", PROMPT_STYLE,
        "--modality-mode", MODALITY_MODE,
        "--save-dtype", SAVE_DTYPE,
        "--gc-every", "5",
        "--skip-existing-complete",
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

In [ ]:
# Merge each split's chunks into one embedding file.
for split in ["train", "dev", "test"]:
    cmd = [
        sys.executable, str(Path(CODE) / "scripts/merge_embedding_chunks.py"),
        "--chunks-dir", str(CHUNK_ROOT / split),
        "--output-pt", str(EMB_ROOT / f"meld_{split}.pt"),
        "--pattern", f"{split}_*.pt",
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

In [ ]:
# Validate merged tensors and extraction metadata before training.
for split in ["train", "dev", "test"]:
    payload = torch.load(EMB_ROOT / f"meld_{split}.pt", map_location="cpu", weights_only=False)
    chunk_configs = payload.get("config", {}).get("merged_from", [])
    assert chunk_configs, f"No chunk configuration recorded for {split}"
    for chunk_entry in chunk_configs:
        config = chunk_entry["config"]
        assert float(config.get("fps")) == FPS, (split, chunk_entry["chunk"], config.get("fps"), FPS)
        assert int(config.get("max_frames")) == MAX_FRAMES
        assert config.get("model_id") == MODEL_ID
        assert config.get("modality_mode") == MODALITY_MODE
        assert config.get("pooling") == POOLING
        assert config.get("prompt_style") == PROMPT_STYLE
    first_config = chunk_configs[0]["config"]
    assert payload["embeddings"].ndim == 2
    assert payload["embeddings"].shape[0] == payload["labels"].shape[0]
    print(
        split,
        "embeddings=", tuple(payload["embeddings"].shape),
        "dtype=", payload["embeddings"].dtype,
        "errors=", len(payload.get("errors", [])),
        "fps=", first_config.get("fps"),
    )

In [ ]:
# Train the same one-hidden-layer MLP probe used for the other VLMs.
cmd = [
    sys.executable, str(Path(CODE) / "scripts/train_mlp.py"),
    "--train-pt", str(EMB_ROOT / "meld_train.pt"),
    "--dev-pt", str(EMB_ROOT / "meld_dev.pt"),
    "--test-pt", str(EMB_ROOT / "meld_test.pt"),
    "--output-dir", str(OUT_DIR),
    "--hidden-dim", "512",
    "--epochs", "50",
    "--device", "cuda",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Main metrics
import json

with open(OUT_DIR / "metrics.json", "r") as f:
    metrics = json.load(f)

for split_name, split_metrics in [("Best dev", metrics["best_dev"]), ("Test", metrics["test"])]:
    print(f"\n{split_name}")
    print("Accuracy:", split_metrics["accuracy"])
    print("Macro F1:", split_metrics["macro_f1"])
    print("Weighted F1:", split_metrics["weighted_f1"])

In [ ]:
# Labeled confusion matrices and per-class reports
DEFAULT_LABELS = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]
labels = metrics.get("label_names", DEFAULT_LABELS)

def labeled_confusion_matrix(split_metrics):
    return pd.DataFrame(
        split_metrics["confusion_matrix"],
        index=[f"true_{label}" for label in labels],
        columns=[f"pred_{label}" for label in labels],
    )

print("Rows are true labels; columns are predicted labels.")
display(labeled_confusion_matrix(metrics["best_dev"]))
display(labeled_confusion_matrix(metrics["test"]))
display(pd.DataFrame(metrics["best_dev"]["classification_report"]).T)
display(pd.DataFrame(metrics["test"]["classification_report"]).T)